# **Problem Statement**

## **Business Context**

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.

To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.

## **Objective**

As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:
- **With Helmet:** Workers wearing safety helmets.
- **Without Helmet:** Workers not wearing safety helmets.

## **Data Description**

The dataset consists of **4125 images**, divided into two categories:

- **With Helmet:** 3161 images showing workers wearing helmets.
- **Without Helmet:** 964 images showing workers not wearing helmets.

**Dataset Characteristics:**
- **Variations in Conditions:** Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
- **Worker Activities:** Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Installing and Importing the Necessary Libraries**

In [ ]:
!pip install tensorflow[and-cuda] scikit-learn==1.6.1 opencv-python==4.12.0.88 seaborn==0.13.2 matplotlib==3.10.0 numpy==2.0.2 pandas==2.2.2 -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
import math

# TensorFlow / Keras
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from keras.applications.vgg16 import VGG16

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    recall_score, precision_score, f1_score
)

# Display settings
pd.set_option("display.max_columns", None)
warnings = __import__("warnings")
warnings.filterwarnings("ignore")


In [ ]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.__version__)

In [ ]:
# 1. Set Python random seed
random.seed(812)

# 2. Set NumPy random seed
np.random.seed(812)

# 3. Set TensorFlow seed (covers Keras + backend)
tf.keras.utils.set_random_seed(812)

# 4. Enable deterministic GPU ops (if using GPU)
tf.config.experimental.enable_op_determinism()

# **Data Overview**


##Loading the data

In [ ]:
# The project data files should be uploaded to the Colab working directory.
# This helper also allows the notebook to run if it is opened from a local Jupyter directory.

image_candidates = ["/content/images.npy", "images.npy"]
label_candidates = ["/content/labels.csv", "labels.csv"]

image_path = next((p for p in image_candidates if os.path.exists(p)), None)
label_path = next((p for p in label_candidates if os.path.exists(p)), None)

if image_path is None or label_path is None:
    raise FileNotFoundError(
        "Could not find images.npy and/or labels.csv. "
        "Upload both files to the Colab session before running this cell."
    )

images = np.load(image_path)
labels = pd.read_csv(label_path)

print("Images shape :", images.shape)
print("Labels shape :", labels.shape)
print("Image dtype  :", images.dtype)
print("Pixel range  :", images.min(), "to", images.max())
print("\nLabel counts:")
print(labels["label"].value_counts().sort_index())


# **Exploratory Data Analysis**

###Plot random images from each of the classes and print their corresponding labels.

In [ ]:
# Plot multiple random examples from both classes.
label_names = {0: "Without Helmet", 1: "With Helmet"}
y_all = labels["label"].astype(int)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row, class_id in enumerate([0, 1]):
    class_indices = np.where(y_all.to_numpy() == class_id)[0]
    chosen = np.random.choice(class_indices, size=4, replace=False)

    for col, idx in enumerate(chosen):
        axes[row, col].imshow(images[idx])
        axes[row, col].set_title(f"{label_names[class_id]} | Label: {class_id}")
        axes[row, col].axis("off")

plt.suptitle("Random Images from Each Helmet-Compliance Class", fontsize=14)
plt.tight_layout()
plt.show()


## Checking for class imbalance


In [ ]:
class_counts = y_all.value_counts().sort_index()
class_percent = (class_counts / len(y_all) * 100).round(2)

class_summary = pd.DataFrame({
    "Class": [label_names[i] for i in class_counts.index],
    "Count": class_counts.values,
    "Percentage": class_percent.values
}, index=class_counts.index)

display(class_summary)

plt.figure(figsize=(7, 4))
ax = sns.countplot(x=y_all, order=[0, 1])
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks([0, 1], ["Without Helmet (0)", "With Helmet (1)"])

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width()/2, p.get_height()),
        ha="center", va="bottom"
    )

plt.tight_layout()
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Majority-to-minority ratio: {imbalance_ratio:.2f}:1")


### **EDA Observations**

- The dataset contains **4,125 RGB images** of size **200 × 200 × 3**, with pixel values in the standard 0–255 range.
- The visual samples show meaningful real-world variation in worker appearance, background, pose, viewing angle, and workplace setting. This makes the task more representative than a highly controlled image dataset.
- The target is **imbalanced**: **3,161 images (76.63%)** belong to the *With Helmet* class, whereas **964 images (23.37%)** belong to the *Without Helmet* class. The majority class is therefore about **3.28 times** the minority class.
- Because of this imbalance, **accuracy alone can be misleading**. In addition to overall accuracy, precision, recall and F1-score, the evaluation will specifically monitor **recall for the Without Helmet class**. Missing a worker who is not wearing a helmet is the more safety-critical error because the system would incorrectly treat a non-compliant situation as compliant.
- The train, validation and test partitions should therefore be **stratified** so that the same class proportions are maintained across all three datasets.


# **Data Preprocessing**

### Splitting the dataset



In [ ]:
# Use a stratified 70% / 15% / 15% split.
# The test set remains untouched until the final model has been selected.

X_train, X_temp, y_train, y_temp = train_test_split(
    images,
    y_all,
    test_size=0.30,
    random_state=812,
    stratify=y_all
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=812,
    stratify=y_temp
)

print("Training set  :", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set      :", X_test.shape, y_test.shape)

split_distribution = pd.DataFrame({
    "Train %": y_train.value_counts(normalize=True).sort_index() * 100,
    "Validation %": y_val.value_counts(normalize=True).sort_index() * 100,
    "Test %": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)

split_distribution.index = ["Without Helmet (0)", "With Helmet (1)"]
print("\nClass proportions after stratified splitting:")
display(split_distribution)


### Data Normalization

In [ ]:
# Normalize image pixels from [0, 255] to [0, 1].
X_train_normalized = X_train.astype("float32") / 255.0
X_val_normalized = X_val.astype("float32") / 255.0
X_test_normalized = X_test.astype("float32") / 255.0

print("Training normalized range  :", X_train_normalized.min(), "to", X_train_normalized.max())
print("Validation normalized range:", X_val_normalized.min(), "to", X_val_normalized.max())
print("Test normalized range      :", X_test_normalized.min(), "to", X_test_normalized.max())


### **Pre-processing Observations**

- A **stratified 70% / 15% / 15% split** is used so that training, validation and test sets retain approximately the same helmet/no-helmet proportions as the full dataset.
- The test set is kept separate from model development and is only used once, after final model selection.
- Pixel values are normalized from **0–255 to 0–1**, which improves numerical stability during neural-network training.
- No data augmentation is applied to validation or test data. Augmentation is introduced only for Model 4 and only on the training set.


# **Model Building**

## Model Evaluation Criterion

The classes are imbalanced, and the minority class (**Without Helmet = 0**) is the most important class from a workplace-safety perspective. A missed non-compliant worker could result in a safety risk being overlooked.

Accordingly, model performance will be assessed using:
- **Accuracy** for overall correctness,
- **Weighted Precision, Recall and F1-score** to summarize performance while accounting for class frequency,
- **Recall for Without Helmet (class 0)** as a safety-focused metric,
- **Confusion matrices** to inspect the types of classification errors.

The **validation set** will be used to compare and select models. The **test set will only be evaluated after final model selection** to provide an unbiased estimate of generalization.


## Utility Functions

In [ ]:
def plot_training_history(history, model_name):
    """Plot accuracy and loss curves separately for clear overfitting/generalization checks."""
    plt.figure(figsize=(7, 4))
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name}: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name}: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Binary Cross-Entropy Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Compute overall and class-specific classification metrics.
    Class 0 = Without Helmet; Class 1 = With Helmet.
    """
    prob = model.predict(predictors, verbose=0).reshape(-1)
    pred = (prob > 0.5).astype(int)
    target_array = np.asarray(target).reshape(-1)

    return pd.DataFrame({
        "Accuracy": [accuracy_score(target_array, pred)],
        "Recall (Weighted)": [recall_score(target_array, pred, average="weighted", zero_division=0)],
        "Precision (Weighted)": [precision_score(target_array, pred, average="weighted", zero_division=0)],
        "F1 Score (Weighted)": [f1_score(target_array, pred, average="weighted", zero_division=0)],
        "Recall - Without Helmet": [recall_score(target_array, pred, pos_label=0, zero_division=0)],
        "Recall - With Helmet": [recall_score(target_array, pred, pos_label=1, zero_division=0)]
    })


In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """Plot a clearly labelled confusion matrix for helmet compliance."""
    prob = model.predict(predictors, verbose=0).reshape(-1)
    pred = (prob > 0.5).astype(int)
    target_array = np.asarray(target).reshape(-1)

    cm = confusion_matrix(target_array, pred, labels=[0, 1])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Without Helmet", "With Helmet"],
        yticklabels=["Without Helmet", "With Helmet"]
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()


##Model 1: Convolutional Neural Network (CNN) from Scratch

In [ ]:
# Model 1: CNN built from scratch
model_1 = Sequential([
    Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(200, 200, 3)),
    MaxPooling2D((4, 4), padding="same"),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2), padding="same"),

    Conv2D(128, (3, 3), activation="relu", padding="same"),

    Flatten(),
    Dense(4, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_1.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_1.summary()

history_1 = model_1.fit(
    X_train_normalized,
    y_train,
    validation_data=(X_val_normalized, y_val),
    epochs=10,
    batch_size=32,
    shuffle=True,
    verbose=1
)

# Training history: inspect both accuracy and loss for overfitting/generalization.
plot_training_history(history_1, "Model 1 - CNN from Scratch")

# Performance
model_1_train_perf = model_performance_classification(model_1, X_train_normalized, y_train)
model_1_valid_perf = model_performance_classification(model_1, X_val_normalized, y_val)

print("Training performance")
display(model_1_train_perf)
plot_confusion_matrix(model_1, X_train_normalized, y_train)

print("Validation performance")
display(model_1_valid_perf)
plot_confusion_matrix(model_1, X_val_normalized, y_val)


### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 1.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_1.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 1 Observations**

**Architecture**
- The first model is a compact CNN trained completely from scratch. It must learn low-level visual patterns and the final helmet/no-helmet decision only from the HelmNet training images.
- This makes it an important **baseline** for judging the value added by transfer learning.

**Training behavior**
- Compare both the **accuracy curves and loss curves**. A widening train-validation gap, especially if validation loss begins to rise while training loss falls, would indicate overfitting.
- Because this model has no pre-trained visual knowledge, slower convergence than VGG-16 models would be expected.

**Performance**
- Overall accuracy alone is not sufficient because the dataset contains many more helmet images than no-helmet images.
- The validation confusion matrix and **Recall - Without Helmet** should therefore receive special attention.
- The most safety-critical error is an actual **Without Helmet** worker being predicted as **With Helmet**, since that can allow non-compliance to go undetected.

**Key insight**
- Model 1 establishes how well a task-specific CNN can perform without transfer learning. The later models should be judged on whether they improve validation generalization and minority-class detection rather than merely fitting the training data better.


## Model 2: Transfer Learning with VGG-16 (Base)

In [ ]:
# Model 2: VGG-16 convolutional base + sigmoid output layer
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(200, 200, 3)
)

# Freeze all pre-trained convolutional layers.
for layer in vgg_base.layers:
    layer.trainable = False

model_2 = Sequential([
    vgg_base,
    Flatten(),
    Dense(1, activation="sigmoid")
])

model_2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_2.summary()

epochs = 10
batch_size = 32
plain_datagen = ImageDataGenerator()

history_2 = model_2.fit(
    plain_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=batch_size,
        seed=812,
        shuffle=True
    ),
    epochs=epochs,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plot_training_history(history_2, "Model 2 - VGG-16 Base")

model_2_train_perf = model_performance_classification(model_2, X_train_normalized, y_train)
model_2_valid_perf = model_performance_classification(model_2, X_val_normalized, y_val)

print("Training performance")
display(model_2_train_perf)
plot_confusion_matrix(model_2, X_train_normalized, y_train)

print("Validation performance")
display(model_2_valid_perf)
plot_confusion_matrix(model_2, X_val_normalized, y_val)


### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 2.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_2.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 2 Observations**

**Architecture**
- Model 2 uses the convolutional part of **VGG-16 pre-trained on ImageNet** as a frozen feature extractor, with a new sigmoid classifier for HelmNet.
- Freezing the VGG-16 layers reduces the number of trainable parameters and allows the model to reuse broadly useful visual representations learned from a much larger image dataset.

**Training behavior**
- Transfer learning should normally converge faster than the scratch CNN because the network does not need to relearn basic visual representations from the beginning.
- The accuracy and loss curves should be checked for stability and for the size of the train-validation gap.

**Performance**
- The important comparison with Model 1 is whether VGG-16 improves validation **Accuracy, Weighted F1 and Recall - Without Helmet**.
- The confusion matrix should be inspected for actual no-helmet workers predicted as helmeted.

**Key insight**
- If Model 2 generalizes better than Model 1, it provides evidence that pre-trained visual features are useful for this task even though VGG-16 was not originally trained specifically for safety-helmet classification.
- Strong validation results are encouraging, but they do not by themselves prove robustness to every real workplace camera condition.


## Model 3: Transfer Learning with VGG-16 (Base + FFNN)





In [ ]:
# Model 3: VGG-16 base + feed-forward neural network (FFNN)
model_3 = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_3.summary()

history_3 = model_3.fit(
    plain_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=32,
        seed=812,
        shuffle=True
    ),
    epochs=10,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plot_training_history(history_3, "Model 3 - VGG-16 Base + FFNN")

model_3_train_perf = model_performance_classification(model_3, X_train_normalized, y_train)
model_3_valid_perf = model_performance_classification(model_3, X_val_normalized, y_val)

print("Training performance")
display(model_3_train_perf)
plot_confusion_matrix(model_3, X_train_normalized, y_train)

print("Validation performance")
display(model_3_valid_perf)
plot_confusion_matrix(model_3, X_val_normalized, y_val)


#### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 3.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_3.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 3 Observations**

**Architecture**
- Model 3 retains the frozen VGG-16 feature extractor and adds a custom **FFNN classifier (128 → 64 → 1)**.
- The dense layers increase task-specific classification capacity, while **Dropout(0.5)** is included to reduce overfitting.

**Training behavior**
- Compare Model 3 with Model 2 using both training and validation curves.
- If training performance improves strongly but validation performance changes little or validation loss worsens, the additional FFNN capacity may be fitting the training set without improving generalization.

**Performance**
- The model should be assessed using validation Accuracy, Weighted F1 and especially **Recall - Without Helmet**.
- A genuine improvement over Model 2 should appear on the validation set, not only on the training set.

**Key insight**
- The VGG-16 base provides general visual feature extraction while the FFNN gives the classifier more flexibility to adapt those features to the helmet-compliance decision.
- The value of this added flexibility depends on the observed validation results; more layers are not automatically better.


## Model 4: Transfer Learning with VGG-16 (Base + FFNN + Data Augmentation)

- In most of the real-world case studies, it is challenging to acquire a large number of images and then train CNNs.
- To overcome this problem, one approach we might consider is **Data Augmentation**.
- CNNs have the property of **translational invariance**, which means they can recognise an object even if its appearance shifts translationally in some way. - Taking this attribute into account, we can augment the images using the techniques listed below

    -  Horizontal Flip (should be set to True/False)
    -  Vertical Flip (should be set to True/False)
    -  Height Shift (should be between 0 and 1)
    -  Width Shift (should be between 0 and 1)
    -  Rotation (should be between 0 and 180)
    -  Shear (should be between 0 and 1)
    -  Zoom (should be between 0 and 1) etc.

Remember, **data augmentation should not be used in the validation/test data set**.

**For HelmNet:** augmentation is applied to the training set only. The chosen transformations represent plausible geometric/positional variation; vertical flipping is avoided because it would create unrealistic worker orientation.

In [ ]:
# Model 4: VGG-16 base + FFNN + training-only data augmentation
model_4 = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_4.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_4.summary()

# Use realistic augmentations for worker images.
# Vertical flipping is intentionally avoided because upside-down workers are not a realistic camera condition.
augment_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

history_4 = model_4.fit(
    augment_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=32,
        seed=812,
        shuffle=True
    ),
    epochs=10,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plot_training_history(history_4, "Model 4 - VGG-16 + FFNN + Data Augmentation")

model_4_train_perf = model_performance_classification(model_4, X_train_normalized, y_train)
model_4_valid_perf = model_performance_classification(model_4, X_val_normalized, y_val)

print("Training performance")
display(model_4_train_perf)
plot_confusion_matrix(model_4, X_train_normalized, y_train)

print("Validation performance")
display(model_4_valid_perf)
plot_confusion_matrix(model_4, X_val_normalized, y_val)


#### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 4.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_4.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 4 Observations**

**Architecture**
- Model 4 combines the frozen VGG-16 feature extractor and FFNN classifier with **data augmentation applied only to training images**.
- Validation and test images remain unchanged, so evaluation is performed on the original data distribution without augmentation leakage.

**Training behavior**
- Rotation, width/height shifts, shear, zoom and horizontal flipping generate plausible variation in worker position, scale and viewpoint.
- Training accuracy can be lower or noisier than Model 3 because the model sees modified images during training. This is not necessarily negative if validation performance remains strong.
- Vertical flipping is intentionally excluded because upside-down workers are not representative of normal workplace camera imagery.

**Performance**
- The central question is whether augmentation improves or stabilizes validation performance, particularly **Recall - Without Helmet**, while maintaining strong overall F1 and accuracy.
- A smaller train-validation gap would provide evidence that augmentation is acting as useful regularization.

**Key insight**
- Data augmentation does not prove robustness to every real-world condition, but it reduces dependence on the exact position/orientation of training images and therefore provides a stronger robustness-oriented design.
- Lighting variation, heavy motion blur and severe occlusion are **not directly created by the current augmentation settings** and should still be validated separately.


### **Qualitative Validation Prediction Review**

In addition to numerical metrics, the following cell displays a small random sample of validation images together with the true label, predicted label and predicted probability.

This does **not** replace full-set evaluation. Its purpose is to visually inspect whether correct and incorrect predictions appear sensible and to identify examples that may warrant further investigation.


In [ ]:
# Qualitative prediction review on a reproducible random validation sample.
rng_review = np.random.default_rng(812)
review_indices = rng_review.choice(len(X_val), size=min(8, len(X_val)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = np.array(axes).reshape(-1)

for ax, idx in zip(axes, review_indices):
    p_helmet = float(model_4.predict(
        X_val_normalized[idx:idx+1], verbose=0
    ).reshape(-1)[0])
    pred_label = int(p_helmet >= 0.5)
    true_label = int(np.asarray(y_val).reshape(-1)[idx])

    ax.imshow(X_val[idx])
    ax.axis("off")
    ax.set_title(
        f"True: {label_names[true_label]}\n"
        f"Pred: {label_names[pred_label]}\n"
        f"P(Helmet)={p_helmet:.3f}",
        fontsize=9
    )

plt.tight_layout()
plt.show()


# **Model Performance Comparison and Final Model Selection**

In [ ]:
# Compare training and validation performance across all four models.
model_names = [
    "CNN from Scratch",
    "VGG-16 Base",
    "VGG-16 Base + FFNN",
    "VGG-16 Base + FFNN + Augmentation"
]

train_perf_list = [
    model_1_train_perf,
    model_2_train_perf,
    model_3_train_perf,
    model_4_train_perf
]

valid_perf_list = [
    model_1_valid_perf,
    model_2_valid_perf,
    model_3_valid_perf,
    model_4_valid_perf
]

models_train_comp_df = pd.concat(train_perf_list, ignore_index=True)
models_train_comp_df.index = model_names

models_valid_comp_df = pd.concat(valid_perf_list, ignore_index=True)
models_valid_comp_df.index = model_names

print("Training Performance Comparison")
display(models_train_comp_df.round(4))

print("\nValidation Performance Comparison")
display(models_valid_comp_df.round(4))

print("\nTrain - Validation Performance Gap")
display((models_train_comp_df - models_valid_comp_df).round(4))

# Safety-first model selection:
# 1) Highest validation recall for Without Helmet
# 2) Highest validation weighted F1
# 3) Highest validation accuracy
ranking = models_valid_comp_df.sort_values(
    by=["Recall - Without Helmet", "F1 Score (Weighted)", "Accuracy"],
    ascending=False
)

top_recall = ranking.iloc[0]["Recall - Without Helmet"]
top_f1 = ranking.iloc[0]["F1 Score (Weighted)"]
top_acc = ranking.iloc[0]["Accuracy"]

# If several models are effectively tied on the three validation criteria,
# prefer the augmented VGG-16 model because it has the strongest robustness-oriented design.
tolerance = 1e-6
tied_models = ranking[
    (np.abs(ranking["Recall - Without Helmet"] - top_recall) <= tolerance) &
    (np.abs(ranking["F1 Score (Weighted)"] - top_f1) <= tolerance) &
    (np.abs(ranking["Accuracy"] - top_acc) <= tolerance)
].index.tolist()

robustness_preference = [
    "VGG-16 Base + FFNN + Augmentation",
    "VGG-16 Base + FFNN",
    "VGG-16 Base",
    "CNN from Scratch"
]

best_model_name = next(
    (name for name in robustness_preference if name in tied_models),
    ranking.index[0]
)

model_lookup = {
    "CNN from Scratch": model_1,
    "VGG-16 Base": model_2,
    "VGG-16 Base + FFNN": model_3,
    "VGG-16 Base + FFNN + Augmentation": model_4
}

best_model = model_lookup[best_model_name]

print("\nSelected Final Model:", best_model_name)
print(
    "Selection basis: validation Recall for Without Helmet first, "
    "then weighted F1 and Accuracy; augmentation/robustness is used only as a tie-breaker."
)


### **Final Model Selection Rationale**

The four models are compared using the **validation set only**. The test set is deliberately excluded from model selection so that it remains an untouched final check.

The ranking follows a safety-first order:

1. **Recall - Without Helmet** — prioritized because missed non-compliance is the most consequential error.
2. **Weighted F1-score** — checks balanced predictive quality while accounting for the class distribution.
3. **Accuracy** — retained as an overall performance measure.
4. If models are effectively tied, preference is given to the augmented VGG-16 model because its training process intentionally exposes the classifier to plausible positional and viewpoint variation.

This approach is more appropriate than selecting a model only because it has the highest training accuracy.

### Technical interpretation of the model progression

- **CNN from Scratch:** establishes the baseline when all visual representations must be learned from HelmNet.
- **VGG-16 Base:** tests whether transfer learning provides better generalization with fewer trainable parameters.
- **VGG-16 + FFNN:** tests whether additional task-specific classifier capacity improves validation performance.
- **VGG-16 + FFNN + Augmentation:** tests whether robustness-oriented training improves or stabilizes generalization.

The code above prints the model selected from the **actual validation metrics**. Only that model is evaluated on the test set below.


## **Validation Threshold Sensitivity Analysis**

The neural network outputs a probability for the **With Helmet** class. A threshold of 0.50 is the standard starting point, but in a safety application the operating threshold may later be adjusted.

Because **class 0 = Without Helmet**, increasing the threshold required to declare *With Helmet* makes the system more conservative: more uncertain cases are treated as possible no-helmet cases. This can improve no-helmet recall, but it may also create more false alerts.

The table below explores several thresholds **using validation data only**. It is included to support deployment planning; the primary project comparison remains based on the standard 0.50 threshold.


In [ ]:
# Threshold sensitivity is evaluated on validation data only.
# This avoids using the test set to tune an operational decision threshold.

val_prob_best = best_model.predict(X_val_normalized, verbose=0).reshape(-1)
y_val_array = np.asarray(y_val).reshape(-1)

threshold_rows = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred_t = (val_prob_best >= threshold).astype(int)

    threshold_rows.append({
        "Threshold for With Helmet": threshold,
        "Accuracy": accuracy_score(y_val_array, pred_t),
        "Weighted F1": f1_score(y_val_array, pred_t, average="weighted", zero_division=0),
        "Recall - Without Helmet": recall_score(
            y_val_array, pred_t, pos_label=0, zero_division=0
        ),
        "Recall - With Helmet": recall_score(
            y_val_array, pred_t, pos_label=1, zero_division=0
        )
    })

threshold_analysis_df = pd.DataFrame(threshold_rows)
display(threshold_analysis_df.round(4))

print(
    "Interpretation: a higher threshold for declaring 'With Helmet' can make the "
    "system more conservative and may increase detection of no-helmet cases, "
    "but it can also increase false alerts. Any production threshold should be "
    "chosen using business risk and site-validation data."
)


## Test Performance

In [ ]:
# Final, one-time evaluation on the untouched test set.
model_test_perf = model_performance_classification(
    best_model,
    X_test_normalized,
    y_test
)

print("Final model:", best_model_name)
print("\nTest performance metrics")
display(model_test_perf.round(4))

print("\nTest confusion matrix")
plot_confusion_matrix(best_model, X_test_normalized, y_test)

# Detailed class-wise report for deployment-oriented interpretation.
test_prob = best_model.predict(X_test_normalized, verbose=0).reshape(-1)
test_pred = (test_prob > 0.5).astype(int)

print("\nDetailed test classification report")
print(
    classification_report(
        np.asarray(y_test),
        test_pred,
        labels=[0, 1],
        target_names=["Without Helmet", "With Helmet"],
        digits=4,
        zero_division=0
    )
)


### **Test-Set Interpretation**

The test set is evaluated **only after the final model has been chosen**, giving the cleanest estimate of performance on unseen images from this dataset.

**What to check**
- Compare test Accuracy and Weighted F1 with validation performance. Similar values suggest stable generalization within the available dataset.
- Inspect **Recall - Without Helmet** and the class-wise classification report.
- In the confusion matrix, focus especially on **Actual Without Helmet → Predicted With Helmet**. These are missed safety violations.

**Important deployment interpretation**
- Very high or even perfect internal test performance should be interpreted cautiously.
- It demonstrates strong performance on the held-out HelmNet sample, but it does **not** establish that the model is automatically production-ready.
- A real monitoring system may encounter camera positions, illumination, PPE styles, occlusion, crowding and image quality that are not fully represented in the project dataset.

Therefore, the final model should be treated as a strong **deployment candidate** that still requires real-site validation before being relied upon for operational safety decisions.


# **Actionable Insights & Recommendations**

### **Key Takeaways**

- The dataset is substantially imbalanced toward **With Helmet** images, so accuracy must be interpreted alongside class-specific metrics.
- **Recall - Without Helmet** is especially important because the highest-risk error is allowing an unsafe no-helmet case to be classified as compliant.
- The model progression isolates the contribution of three ideas: learning from scratch, **transfer learning**, added FFNN capacity, and **data augmentation**.
- VGG-16 provides reusable pre-trained visual representations; the observed validation results determine whether this actually improves HelmNet performance.
- Data augmentation introduces plausible positional and geometric variation, but does not automatically cover difficult conditions such as severe low light, motion blur or heavy occlusion.
- The final model is selected from validation results, then evaluated once on the untouched test set.
- Threshold sensitivity shows that the operating decision rule can later be tuned to place greater emphasis on missed safety violations.

### **Recommendations for SafeGuard Corp**

1. **Use the model initially as safety decision support.** Integrate it with existing workplace safety procedures rather than making it the sole compliance control.

2. **Track the safety-critical miss rate.** A key operational KPI should be the number/rate of actual *Without Helmet* cases classified as *With Helmet*, alongside accuracy, precision and F1-score.

3. **Tune the production threshold using site-specific risk.** The default 0.50 threshold is appropriate for model comparison, but deployment can use validation/site data to determine whether a more conservative threshold better balances missed violations and false alerts.

4. **Use confidence-aware workflows.** High-confidence detections can trigger automated alerts, while ambiguous/low-confidence cases can be routed for human review.

5. **Run a real-site pilot before broad deployment.** Evaluate CCTV or camera images containing low illumination, shadows/glare, unusual angles, distant workers, partial occlusion, crowding, different helmet styles/colours, caps/hoods and varying image quality.

6. **Expand the minority-class dataset.** Future collection should deliberately add more *Without Helmet* examples, especially difficult real-world cases, because this is the smaller and more safety-critical class.

7. **Monitor model drift.** Recalculate class-wise recall, precision, F1-score and confusion matrices as sites, cameras, uniforms, PPE and working conditions change.

8. **Consider optimized edge inference after validation.** If real-time monitoring requires low latency or reduced video transfer, evaluate an optimized deployment format such as TensorFlow Lite or another suitable inference runtime and benchmark both latency and predictive performance before rollout.

9. **Extend to broader PPE monitoring only after helmet detection is validated.** The same computer-vision approach could later support safety-vest, eyewear or other PPE compliance use cases.

### **Conclusion**

The HelmNet project develops a complete computer-vision workflow for helmet-compliance classification, progressing from a CNN trained from scratch to VGG-16 transfer-learning models with custom classification layers and data augmentation.

The strongest model should be selected from the **actual validation results**, with priority given to detecting workers **without helmets**, and then confirmed on the untouched test set. Strong project-dataset performance demonstrates technical feasibility, while real-site testing, threshold calibration, human-review workflows and ongoing monitoring are necessary steps before production safety use.

This aligns the modelling strategy with SafeGuard Corp's core objective: **scalable monitoring that reduces the chance of helmet non-compliance going undetected.**


<font size=5 color='blue'>Power Ahead!</font>
___